In [ ]:
import os,sys
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scanpy import AnnData
import mudata
from scipy.sparse import csr_matrix
import warnings
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import harmonypy as hm

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
sc.settings.verbosity = 3
warnings.filterwarnings("ignore")
plt.rcParams['pdf.fonttype']=42

In [ ]:
#third trimester

snm3 = sc.read_h5ad("/u/project/cluo/heffel/BICAN3/DATA/All_3T_18030x49367.h5ad")
snm3

In [ ]:
#600 umi filtered

bgs5 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260107_bgs5_600_umi_filtered_normalized.h5ad")
bgs5

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Get the cluster annotation
clusters = bgs5.obs['leiden']

# Convert to string for hue (safe handling)
cluster_str = clusters.astype(str)

# Identify numeric clusters for sorting
numeric = pd.to_numeric(clusters, errors='coerce')
is_numeric = numeric.notna()
numeric_unique = np.sort(numeric[is_numeric].unique().astype(int))

# Non-numeric clusters
non_numeric_unique = np.sort(clusters[~is_numeric].unique())

# Full sorted list: numerics first (as str), then non-numerics
all_unique = [str(c) for c in numeric_unique] + [str(c) for c in non_numeric_unique]

# Color palette (tab20 has 20 colors; increase n if you have more clusters)
palette = sns.color_palette('tab20', len(all_unique))
cluster_colors = {clust: palette[i] for i, clust in enumerate(all_unique)}

# Spatial coordinates - prefer obsm['spatial'] if present
if 'spatial' in bgs5.obsm:
    spatial_coords = bgs5.obsm['spatial']
    x_spatial, y_spatial = spatial_coords[:, 0], spatial_coords[:, 1]
else:
    x_spatial = bgs5.obs['CenterX_global_px']
    y_spatial = bgs5.obs['CenterY_global_px']

# UMAP coordinates
umap_coords = bgs5.obsm['X_umap']

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Spatial plot
sns.scatterplot(x=x_spatial, y=y_spatial,
                hue=cluster_str, palette=cluster_colors,
                s=2, edgecolor=None, ax=axes[0], legend=False)
axes[0].set_title('Adjusted L3 Clusters - Spatial')
axes[0].set_xlabel('X (px)')
axes[0].set_ylabel('Y (px)')
axes[0].invert_yaxis()  # Important for aligning with original images
axes[0].set_aspect('equal')

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=cluster_str, palette=cluster_colors,
                s=5, edgecolor=None, ax=axes[1], legend=False)
axes[1].set_title('Adjusted L3 Clusters - UMAP')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

# Shared legend outside
handles = [plt.Line2D([0], [0], marker='o', color='w',
                      markerfacecolor=cluster_colors[clust],
                      markersize=10) for clust in all_unique]
fig.legend(handles=handles, labels=all_unique,
           title='Adjusted L3 Clusters',
           bbox_to_anchor=(1.02, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

In [ ]:
kept_L2 = ['Glial', 'MSN', 'MGE', 'NN', 'RG-1']

mask = snm3.obs['newL2'].isin(kept_L2)

snm3_filt = snm3[mask, :]

In [ ]:
snm3_filt.X = snm3_filt.X * -1

In [ ]:
sc.pp.scale(snm3_filt)

In [ ]:
sc.pp.scale(bgs5)

In [ ]:
harmonized = bgs5.concatenate(snm3_filt)

In [ ]:
sc.tl.pca(harmonized)

In [ ]:
sc.external.pp.harmony_integrate(harmonized, 'batch', max_iter_harmony=20)

In [ ]:
sc.pp.neighbors(harmonized, n_pcs=20, n_neighbors=20, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(harmonized)

In [ ]:
sc.pl.umap(harmonized, color = ['leiden'], legend_loc = 'on data')

#bgs5_filtered

In [ ]:
import scanpy as sc
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd

def knn_label_transfer_from_harmony(
    integrated_adata,
    target_adata,
    label_key='cell_type',
    batch_key='batch',
    target_batch=None,
    harmony_key='X_pca_harmony',
    n_neighbors=15,
    use_rep='X_pca',
    confidence_threshold=0.5,
    return_probabilities=True
):
    """
    Transfer labels from harmony-integrated object to original object using KNN.

    Parameters:
    -----------
    integrated_adata : AnnData
        Concatenated and harmony-integrated object with labels
    target_adata : AnnData
        Original object to receive transferred labels
    label_key : str
        Key in integrated_adata.obs containing labels to transfer
    batch_key : str
        Key identifying different batches/samples
    target_batch : str
        Batch identifier for the target dataset in integrated object
    harmony_key : str
        Key for harmony-corrected embedding in integrated_adata.obsm
    n_neighbors : int
        Number of neighbors for KNN classifier
    use_rep : str
        Representation to use in target_adata (will be computed if needed)
    confidence_threshold : float
        Minimum probability to assign label (otherwise 'uncertain')
    return_probabilities : bool
        Whether to return prediction probabilities

    Returns:
    --------
    target_adata with transferred labels in .obs
    """

    # Get harmony-corrected embedding from integrated object
    if harmony_key not in integrated_adata.obsm:
        raise ValueError(f"{harmony_key} not found in integrated_adata.obsm")

    X_harmony = integrated_adata.obsm[harmony_key]

    # Identify reference cells (all cells except target batch)
    if target_batch is None:
        # If not specified, assume target_adata represents a specific batch
        # Use all cells in integrated object as reference
        ref_mask = np.ones(integrated_adata.n_obs, dtype=bool)
        target_mask = np.zeros(integrated_adata.n_obs, dtype=bool)
        print("Warning: target_batch not specified. Using all integrated cells as reference.")
        print("For proper transfer, specify which batch in integrated object corresponds to target_adata")
    else:
        ref_mask = integrated_adata.obs[batch_key] != target_batch
        target_mask = integrated_adata.obs[batch_key] == target_batch

    # Get reference data and labels
    X_ref = X_harmony[ref_mask]
    y_ref = integrated_adata.obs[label_key].values[ref_mask]

    # Get target cells from integrated object (for matching)
    X_target_integrated = X_harmony[target_mask]

    # Train KNN classifier on reference cells
    print(f"Training KNN classifier with {n_neighbors} neighbors...")
    knn = KNeighborsClassifier(n_neighbors=n_neighbors, weights='distance')
    knn.fit(X_ref, y_ref)

    # Predict labels for target cells in integrated space
    y_pred = knn.predict(X_target_integrated)
    y_proba = knn.predict_proba(X_target_integrated)

    # Get maximum probability for each prediction
    max_proba = y_proba.max(axis=1)

    # Apply confidence threshold
    y_pred_confident = y_pred.copy()
    uncertain_mask = max_proba < confidence_threshold
    if uncertain_mask.sum() > 0:
        y_pred_confident[uncertain_mask] = 'uncertain'
        print(f"Marked {uncertain_mask.sum()} cells as uncertain (confidence < {confidence_threshold})")

    # Match cells between integrated object and original target object
    # This assumes same order or we can match by cell barcodes
    if target_adata.n_obs == target_mask.sum():
        # Same number of cells - assume same order
        target_adata.obs[f'{label_key}_transferred'] = y_pred_confident
        target_adata.obs[f'{label_key}_confidence'] = max_proba

        if return_probabilities:
            # Store full probability matrix
            proba_df = pd.DataFrame(
                y_proba,
                index=target_adata.obs_names,
                columns=[f'{label_key}_prob_{cls}' for cls in knn.classes_]
            )
            for col in proba_df.columns:
                target_adata.obs[col] = proba_df[col].values
    else:
        print(f"Warning: target_adata has {target_adata.n_obs} cells but "
              f"found {target_mask.sum()} cells with batch={target_batch} in integrated object")
        print("Cell matching may be incorrect. Consider matching by cell barcodes.")

    print(f"\nLabel transfer complete!")
    print(f"Transferred labels stored in: {label_key}_transferred")
    print(f"Confidence scores stored in: {label_key}_confidence")

    # Print summary statistics
    print(f"\nTransferred label distribution:")
    print(target_adata.obs[f'{label_key}_transferred'].value_counts())
    print(f"\nMean confidence: {max_proba.mean():.3f}")
    print(f"Median confidence: {np.median(max_proba):.3f}")

    return target_adata

# Example usage:

# Load your integrated object (already harmony-corrected)
# integrated = sc.read_h5ad('integrated_harmony.h5ad')

# Load one of the original objects
# original = sc.read_h5ad('original_sample.h5ad')

# Transfer labels
# original = knn_label_transfer_from_harmony(
#     integrated_adata=integrated,
#     target_adata=original,
#     label_key='cell_type',
#     batch_key='sample',
#     target_batch='sample1',  # identifier for this sample in integrated object
#     harmony_key='X_pca_harmony',
#     n_neighbors=15,
#     confidence_threshold=0.5
# )

# Visualize transferred labels
# sc.pl.umap(original, color=['cell_type_transferred', 'cell_type_confidence'])

In [ ]:
# If you have integrated object with harmony correction
knn_label_transfer_from_harmony(
    integrated_adata=harmonized,  # your harmony-corrected concatenated object
    target_adata=bgs5,  # your original CosMx spatial data
    label_key='adjusted_L3',
    batch_key='batch',
    target_batch='0',  # whatever identifier you used
    harmony_key='X_pca_harmony',
    n_neighbors=20,  # adjust based on your data
    confidence_threshold=0.6
)

# Now your spatial object has transferred labels you can visualize
sc.pl.umap(bgs5, color='adjusted_L3_transferred')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

adata = bgs5
categ = 'adjusted_L3_transferred'

# Get unique clusters
clusters = sorted(adata.obs[categ].unique())
n_clusters = len(clusters)

# Grid setup
ncols = 4
nrows = int(np.ceil(n_clusters / ncols))
figsize = (6 * ncols, 6 * nrows)

# Create figure
fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
axes = axes.flatten() if n_clusters > 1 else [axes]

# Get spatial coordinates
spatial_coords = adata.obsm['spatial']

# Get colors for clusters
if f'{categ}_colors' in adata.uns:
    cluster_colors = dict(zip(adata.obs[categ].cat.categories, adata.uns[f'{categ}_colors']))
else:
    from matplotlib import cm
    cluster_colors = {c: cm.tab20(i % 20) for i, c in enumerate(clusters)}

grey_color = '#D3D3D3'
spot_size = 0.5

# Plot each cluster
for idx, cluster in enumerate(clusters):
    ax = axes[idx]

    # Create mask for this cluster
    mask = adata.obs[categ] == cluster

    # Plot grey background (all other cells)
    ax.scatter(
        spatial_coords[~mask, 0],
        spatial_coords[~mask, 1],
        c=grey_color,
        s=spot_size,
        alpha=0.5
    )

    # Plot this cluster in color
    ax.scatter(
        spatial_coords[mask, 0],
        spatial_coords[mask, 1],
        c=cluster_colors[cluster],
        s=spot_size,
        alpha=1.0
    )

    ax.set_title(f'Leiden {cluster}')
    ax.set_aspect('equal')
    ax.axis('off')

# Remove empty subplots
for idx in range(n_clusters, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
#plt.savefig('individual_clusters_spatial.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os

outdir = "/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/"
os.makedirs(outdir, exist_ok=True)

# Save AnnData objects
bgs5.write(os.path.join(outdir, "20260108_bgs5_filtered_snm3_3T_L3.h5ad"))

In [ ]:
sc.pl.umap(harmonized, color = ['leiden'], legend_loc = 'on data')

In [ ]:
# Rename batch categories for plotting
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

sc.pl.umap(
    harmonized,
    color='batch_named',
    size=5,
    palette={
        'CosMx': 'tab:lightblue',
        'snm3C': 'tab:orange'
    }
)

In [ ]:
# Rename batch categories for plotting
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

sc.pl.umap(
    harmonized,
    color='batch_named',
    size=1.5,
    palette={
        'CosMx': '#9f78de',
        'snm3C': '#70e653'
    }
)

In [ ]:
# Rename batch categories for plotting
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

sc.pl.umap(
    harmonized,
    color='batch_named',
    size=1.5,
    palette={
    'CosMx': '#2F6B9A',   # deep cool blue
    'snm3C': '#E15759'    # desaturated coral
    }
)

In [ ]:
# Rename batch categories for plotting
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

sc.pl.umap(
    harmonized,
    color='batch_named',
    size=1.5,
    palette={
    'CosMx': '#5DA5DA',   # soft sky blue
    'snm3C': '#59A14F'    # muted green
    }
)

In [ ]:
# Rename batch categories for plotting
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

sc.pl.umap(
    harmonized,
    color='batch_named',
    size=1.5,
    palette={
    'CosMx': '#3A5F7D',   # deep slate blue
    'snm3C': '#C97A3A'    # warm bronze
    }
)

# Deep & Elegant (journal-ready, serious tone)

In [ ]:
# Rename batch categories for plotting
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

sc.pl.umap(
    harmonized,
    color='batch_named',
    size=1.5,
    palette={
    'CosMx': '#1F77B4',   # classic but refined blue
    'snm3C': '#D62728'    # muted crimson
    }
)

# Minimal + Punch (high clarity, bold)

In [ ]:
# Rename batch categories for plotting
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

sc.pl.umap(
    harmonized,
    color='batch_named',
    size=1.5,
    palette={
    'CosMx': '#8DA0CB',   # lavender-blue
    'snm3C': '#FC8D62'    # soft peach
    }
)

# Soft Contrast (modern, gentle, Nature-ish)

In [ ]:
# Rename batch categories for plotting
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

sc.pl.umap(
    harmonized,
    color='batch_named',
    size=1.5,
    palette={
    'CosMx': '#5DA5DA',
    'snm3C': '#D62728'
    }
)

#contrast

In [ ]:
#shuffled order --> dont want

import numpy as np
import scanpy as sc

# Rename batch categories for plotting
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Shuffle plotting order (reproducible with a seed)
rng = np.random.default_rng(0)
perm = rng.permutation(harmonized.n_obs)

# Plot using the shuffled view
sc.pl.umap(
    harmonized[perm],
    color='batch_named',
    size=0.5,
    palette={
        'CosMx': 'tab:blue',
        'snm3C': 'tab:orange'
    }
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.lines import Line2D

# Per-celltype highlight plots (Spatial left, HARMONIZED UMAP right)
# - Uses harmonized UMAP (computed on harmonized = bgs5.concatenate(snm3_filt))
# - Shows ONLY CosMx batch cells in UMAP (hides snm3_filt)
# - Colors by bgs5.obs['adjusted_L3_transferred']
# - Keeps your "highlight one type vs others (grey)" layout
# REQUIREMENTS / ASSUMPTIONS:
# 1) harmonized.obs[batch_key] exists and CosMx batch == target_batch (e.g. "0")
# 2) harmonized.obs_names for the CosMx portion match bgs5.obs_names
#    If they don't, we fall back to matching via id_col (cell_id / cell_ID).

# Use your objects
adata = bgs5
harm = harmonized

# Settings (edit if needed)
color_key = 'adjusted_L3_transferred'  # column in bgs5.obs
batch_key = 'batch'
target_batch = '0'                     # CosMx batch in harmonized.obs[batch_key]
id_col = 'cell_id'                     # used only if obs_names don't match

default_color = 'lightgray'

# Get harmonized UMAP coords for ONLY CosMx cells, aligned to adata (bgs5)
if 'X_umap' not in harm.obsm:
    raise ValueError("harmonized.obsm['X_umap'] not found. Run sc.tl.umap(harmonized) first.")

harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]

# Try the cleanest alignment: obs_names identical (often true if you didn't alter names)
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

if np.array_equal(harm_cosmx_names.values, adata.obs_names.values):
    # perfect: same order
    umap_coords = harm_umap_cosmx
else:
    # Fallback: match by cell_id (or cell_ID) between bgs5 and harmonized CosMx portion
    if id_col not in adata.obs.columns or id_col not in harm.obs.columns:
        raise ValueError(
            "CosMx cells in harmonized do not match bgs5.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match or set id_col to a shared unique ID (e.g. 'cell_ID')."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    bgs5_obs = adata.obs[[id_col]].copy()
    bgs5_obs['__bgs5_i'] = np.arange(bgs5_obs.shape[0])

    merged = bgs5_obs.merge(
        harm_cosmx_obs,
        left_on=id_col, right_on=id_col,
        how='inner'
    )

    if merged.shape[0] != adata.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata.n_obs:,} bgs5 cells to harmonized CosMx cells via {id_col}. "
            "Unmatched cells will be dropped from the UMAP plot."
        )

    # Build UMAP coords aligned to merged order (same as merged rows)
    umap_coords = harm_umap_cosmx[merged['__umap_i'].values, :]

    # Also restrict spatial + labels to matched subset to keep masks consistent
    adata = adata[merged['__bgs5_i'].values].copy()

# Spatial coords (from bgs5)
if 'spatial' in adata.obsm:
    spatial_coords = adata.obsm['spatial']
    x_spatial, y_spatial = spatial_coords[:, 0], spatial_coords[:, 1]
else:
    x_spatial = adata.obs['CenterX_global_px'].to_numpy()
    y_spatial = adata.obs['CenterY_global_px'].to_numpy()

# Labels / colors
clusters = adata.obs[color_key].astype(str)
cluster_str = clusters.astype(str)

# Unique cell types (sorted)
unique_clusters = np.sort(cluster_str.unique())

# Pre-defined colors if available
if (color_key in adata.obs and
    adata.obs[color_key].dtype.name == 'category' and
    f'{color_key}_colors' in adata.uns):

    categories = adata.obs[color_key].cat.categories.astype(str)
    color_list = adata.uns[f'{color_key}_colors']
    cluster_colors = dict(zip(categories, color_list))
    print(f"Using pre-defined colors from adata.uns['{color_key}_colors']")
else:
    palette = sns.color_palette('tab20', len(unique_clusters))
    cluster_colors = dict(zip(unique_clusters, palette))

# Loop over each cell type
for clust in unique_clusters:
    highlight_color = cluster_colors.get(clust, 'red')

    # Masks (aligned between spatial + harmonized UMAP coords)
    is_target = (cluster_str == clust).values
    is_background = ~is_target

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # Spatial plot
    ax = axes[0]

    ax.scatter(
        x_spatial[is_background], y_spatial[is_background],
        c=default_color, s=1, edgecolor=None, alpha=0.8, linewidth=0
    )

    ax.scatter(
        x_spatial[is_target], y_spatial[is_target],
        c=highlight_color, s=1.5, edgecolor=None, alpha=1.0, linewidth=0
    )

    ax.set_title(f"Spatial: {clust}", fontsize=14)
    ax.set_xlabel("X (px)")
    ax.set_ylabel("Y (px)")
    ax.invert_yaxis()
    ax.set_aspect("equal")

    # HARMONIZED UMAP plot (CosMx cells only)
    ax = axes[1]

    ax.scatter(
        umap_coords[is_background, 0], umap_coords[is_background, 1],
        c=default_color, s=1, edgecolor=None, alpha=0.7, linewidth=0
    )

    ax.scatter(
        umap_coords[is_target, 0], umap_coords[is_target, 1],
        c=highlight_color, s=2, edgecolor='black', linewidth=0.5, alpha=1.0
    )

    ax.set_title(f"Harmony UMAP (CosMx only): {clust}", fontsize=14)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

    # Legend
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label=clust,
               markerfacecolor=highlight_color, markersize=10, markeredgecolor='black'),
        Line2D([0], [0], marker='o', color='w', label='Other cells',
               markerfacecolor=default_color, markersize=10),
    ]
    ax.legend(handles=legend_elements, loc='upper right', frameon=True)

    plt.suptitle(f"{color_key}: {clust}", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# GRID FIGURE (tighter layout, 2 cell types per row)
# - Per cell type: Spatial (left) + Harmonized UMAP (right)
# - Uses harmonized UMAP (from harmonized = bgs5.concatenate(snm3_filt))
# - Shows ONLY CosMx batch cells in UMAP (hides snm3_filt)
# - Colors by bgs5.obs['adjusted_L3_transferred']
# - Hides these cell types entirely:
#     SST, VLMC, PVALB, ChC, uncertain
# REQUIREMENTS / ASSUMPTIONS:
# 1) harmonized.obs[batch_key] exists and CosMx batch == target_batch (e.g. "0")
# 2) harmonized.obsm['X_umap'] exists
# 3) CosMx cells in harmonized align to bgs5 either by obs_names (best)
#    or by id_col (fallback)

# Use your objects
adata = bgs5
harm = harmonized

# Settings (edit if needed)
color_key = 'adjusted_L3_transferred'   # column in bgs5.obs
batch_key = 'batch'
target_batch = '0'                      # CosMx batch in harmonized.obs[batch_key]
id_col = 'cell_id'                      # fallback matching key if obs_names don't match

exclude_types = {'SST', 'VLMC', 'PVALB', 'ChC', 'uncertain'}

default_color = 'lightgray'

# Sanity checks
if 'X_umap' not in harm.obsm:
    raise ValueError("harmonized.obsm['X_umap'] not found. Run sc.tl.umap(harmonized) first.")

if color_key not in adata.obs.columns:
    raise KeyError(f"{color_key} not found in bgs5.obs")

if batch_key not in harm.obs.columns:
    raise KeyError(f"{batch_key} not found in harmonized.obs")

# Restrict harmonized to CosMx batch (UMAP coordinates for CosMx cells only)
harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

# Align harmonized CosMx UMAP rows to bgs5 cells
aligned_ok = False

# Best case: obs_names match perfectly (including order)
if np.array_equal(harm_cosmx_names.values, adata.obs_names.values):
    umap_coords = harm_umap_cosmx
    aligned_ok = True
else:
    # Fallback: match by id_col
    if id_col not in adata.obs.columns or id_col not in harm.obs.columns:
        raise ValueError(
            "CosMx cells in harmonized do not match bgs5.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match OR set id_col to a shared unique ID (e.g. 'cell_ID')."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    bgs5_obs = adata.obs[[id_col]].copy()
    bgs5_obs['__bgs5_i'] = np.arange(bgs5_obs.shape[0])

    merged = bgs5_obs.merge(harm_cosmx_obs, on=id_col, how='inner')

    if merged.shape[0] == 0:
        raise ValueError(
            f"No cells matched between bgs5 and harmonized CosMx cells using id_col='{id_col}'. "
            "Try switching id_col to 'cell_ID' (or whichever is truly unique and shared)."
        )

    if merged.shape[0] != adata.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata.n_obs:,} bgs5 cells to harmonized CosMx cells via '{id_col}'. "
            "Proceeding with matched subset only."
        )

    umap_coords = harm_umap_cosmx[merged['__umap_i'].values, :]
    adata = adata[merged['__bgs5_i'].values].copy()
    aligned_ok = True

if not aligned_ok:
    raise RuntimeError("Failed to align harmonized UMAP coordinates to bgs5 cells.")

# Filter out excluded cell types (from BOTH spatial and UMAP)
mask_keep = ~adata.obs[color_key].astype(str).isin(exclude_types)
adata = adata[mask_keep].copy()
umap_coords = umap_coords[mask_keep.values, :]

# Spatial coords (from bgs5)
if 'spatial' in adata.obsm:
    spatial_coords = adata.obsm['spatial']
    x_spatial, y_spatial = spatial_coords[:, 0], spatial_coords[:, 1]
else:
    x_spatial = adata.obs['CenterX_global_px'].to_numpy()
    y_spatial = adata.obs['CenterY_global_px'].to_numpy()

# Labels / colors
clusters = adata.obs[color_key].astype(str)
cluster_str = clusters.astype(str)

# Unique cell types (sorted)
unique_clusters = np.sort(cluster_str.unique())

# Pre-defined colors if available
if (
    color_key in adata.obs and
    adata.obs[color_key].dtype.name == 'category' and
    f'{color_key}_colors' in adata.uns
):
    categories = adata.obs[color_key].cat.categories.astype(str)
    color_list = adata.uns[f'{color_key}_colors']
    cluster_colors = dict(zip(categories, color_list))
    print(f"Using pre-defined colors from adata.uns['{color_key}_colors']")
else:
    palette = sns.color_palette('tab20', len(unique_clusters))
    cluster_colors = dict(zip(unique_clusters, palette))

# Grid layout: 2 cell types per row
# Each cell type = 2 columns (Spatial, UMAP) => 4 columns total
n_types = len(unique_clusters)
types_per_row = 2
n_rows = math.ceil(n_types / types_per_row)

fig, axes = plt.subplots(
    n_rows,
    4,
    figsize=(16, 4 * n_rows),
    gridspec_kw={'wspace': 0.15, 'hspace': 0.25}
)

# Make axes always 2D
if n_rows == 1:
    axes = axes[np.newaxis, :]

# Plot each cell type
for i, clust in enumerate(unique_clusters):
    row = i // 2
    col_offset = (i % 2) * 2  # 0 or 2

    highlight_color = cluster_colors.get(clust, 'red')

    is_target = (cluster_str == clust).values
    is_background = ~is_target

    # Spatial
    ax_sp = axes[row, col_offset]
    ax_sp.scatter(
        x_spatial[is_background], y_spatial[is_background],
        c=default_color, s=0.8, alpha=0.7, linewidth=0
    )
    ax_sp.scatter(
        x_spatial[is_target], y_spatial[is_target],
        c=highlight_color, s=1.2, alpha=1.0, linewidth=0
    )
    ax_sp.set_title(clust, fontsize=11)
    ax_sp.set_xticks([])
    ax_sp.set_yticks([])
    ax_sp.invert_yaxis()
    ax_sp.set_aspect('equal')

    if col_offset == 0:
        ax_sp.set_ylabel("Spatial", fontsize=10)

    # Harmonized UMAP (CosMx only)
    ax_um = axes[row, col_offset + 1]
    ax_um.scatter(
        umap_coords[is_background, 0], umap_coords[is_background, 1],
        c=default_color, s=0.8, alpha=0.6, linewidth=0
    )
    ax_um.scatter(
        umap_coords[is_target, 0], umap_coords[is_target, 1],
        c=highlight_color, s=1.5, alpha=1.0, linewidth=0
    )
    ax_um.set_xticks([])
    ax_um.set_yticks([])

    if col_offset == 0:
        ax_um.set_ylabel("Harmony UMAP", fontsize=10)

# Hide unused axes (if odd number of cell types)
for j in range(n_types * 2, n_rows * 4):
    r = j // 4
    c = j % 4
    axes[r, c].axis('off')

plt.suptitle(
    "CosMx Spatial (left) and Harmonized UMAP (right)\nHighlighted by transferred cell type",
    fontsize=16,
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.lines import Line2D

# SINGLE FIGURE:
# - Left: CosMx spatial (bgs5) colored by transferred cell types
# - Right: Harmonized JOINT embedding UMAP (CosMx cells only), same colors
# - Excludes these cell types entirely:
#     SST, VLMC, PVALB, ChC, uncertain
# REQUIREMENTS / ASSUMPTIONS:
# 1) harmonized.obsm['X_umap'] exists (from sc.tl.umap(harmonized))
# 2) harmonized.obs[batch_key] exists and CosMx batch == target_batch (e.g. "0")
# 3) CosMx cells in harmonized align to bgs5 either by obs_names (best)
#    or by id_col (fallback)

# Objects
adata = bgs5
harm = harmonized

# Settings (edit if needed)
color_key = 'adjusted_L3_transferred'
batch_key = 'batch'
target_batch = '0'          # CosMx batch label inside harmonized.obs[batch_key]
id_col = 'cell_id'          # fallback matching key if obs_names don't match
exclude_types = {'SST', 'VLMC', 'PVALB', 'ChC', 'uncertain'}

# Plot styling
bg_color = 'lightgrey'
spatial_s = 1.0
umap_s = 0.6
alpha = 0.85

# Sanity checks
if 'X_umap' not in harm.obsm:
    raise ValueError("harmonized.obsm['X_umap'] not found. Run sc.tl.umap(harmonized) first.")
if color_key not in adata.obs.columns:
    raise KeyError(f"{color_key} not found in bgs5.obs")
if batch_key not in harm.obs.columns:
    raise KeyError(f"{batch_key} not found in harmonized.obs")

# Restrict harmonized to CosMx batch (UMAP coords for CosMx only)
harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

# Align harmonized CosMx UMAP rows to bgs5 cells
if np.array_equal(harm_cosmx_names.values, adata.obs_names.values):
    umap_coords = harm_umap_cosmx
else:
    # Fallback: match by id_col
    if id_col not in adata.obs.columns or id_col not in harm.obs.columns:
        raise ValueError(
            "CosMx cells in harmonized do not match bgs5.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match OR set id_col to a shared unique ID (e.g. 'cell_ID')."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    bgs5_obs = adata.obs[[id_col]].copy()
    bgs5_obs['__bgs5_i'] = np.arange(bgs5_obs.shape[0])

    merged = bgs5_obs.merge(harm_cosmx_obs, on=id_col, how='inner')

    if merged.shape[0] == 0:
        raise ValueError(
            f"No cells matched between bgs5 and harmonized CosMx cells using id_col='{id_col}'. "
            "Try switching id_col to 'cell_ID' (or whichever is truly unique and shared)."
        )

    if merged.shape[0] != adata.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata.n_obs:,} bgs5 cells to harmonized CosMx cells via '{id_col}'. "
            "Proceeding with matched subset only."
        )

    umap_coords = harm_umap_cosmx[merged['__umap_i'].values, :]
    adata = adata[merged['__bgs5_i'].values].copy()

# Filter out excluded cell types (apply to BOTH plots)
labels_all = adata.obs[color_key].astype(str)
mask_keep = ~labels_all.isin(exclude_types)

adata = adata[mask_keep].copy()
umap_coords = umap_coords[mask_keep.values, :]

labels = adata.obs[color_key].astype(str)

# Colors (prefer pre-defined)
unique_labels = np.sort(labels.unique())

if (
    color_key in adata.obs and
    adata.obs[color_key].dtype.name == 'category' and
    f'{color_key}_colors' in adata.uns
):
    categories = adata.obs[color_key].cat.categories.astype(str)
    color_list = adata.uns[f'{color_key}_colors']
    label_colors_all = dict(zip(categories, color_list))
    # Restrict mapping to labels actually present
    label_colors = {k: label_colors_all.get(k, bg_color) for k in unique_labels}
    print(f"Using pre-defined colors from adata.uns['{color_key}_colors']")
else:
    palette = sns.color_palette('tab20', len(unique_labels))
    label_colors = dict(zip(unique_labels, palette))

# Spatial coords
if 'spatial' in adata.obsm:
    xy = adata.obsm['spatial']
    x_sp, y_sp = xy[:, 0], xy[:, 1]
else:
    x_sp = adata.obs['CenterX_global_px'].to_numpy()
    y_sp = adata.obs['CenterY_global_px'].to_numpy()

# Plot: single figure, 2 panels
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Spatial (left)
sns.scatterplot(
    x=x_sp, y=y_sp,
    hue=labels, palette=label_colors,
    s=spatial_s, edgecolor=None, ax=axes[0], legend=False, alpha=alpha
)
axes[0].set_title("CosMx Spatial (transferred cell types)")
axes[0].set_xlabel("X (px)")
axes[0].set_ylabel("Y (px)")
axes[0].invert_yaxis()
axes[0].set_aspect("equal")

# Harmonized UMAP (right) (CosMx only)
sns.scatterplot(
    x=umap_coords[:, 0], y=umap_coords[:, 1],
    hue=labels, palette=label_colors,
    s=umap_s, edgecolor=None, ax=axes[1], legend=False, alpha=alpha
)
axes[1].set_title("Harmonized Joint UMAP (CosMx cells only)")
axes[1].set_xlabel("UMAP 1")
axes[1].set_ylabel("UMAP 2")

# Shared legend outside
handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=label_colors[l], markersize=8, linestyle='None')
    for l in unique_labels
]
fig.legend(
    handles=handles, labels=unique_labels,
    title=color_key,
    bbox_to_anchor=(1.02, 0.5), loc='center left'
)

plt.tight_layout(rect=[0, 0, 0.86, 1])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

# SINGLE FIGURE:
# - Left: CosMx spatial (bgs5) colored by transferred cell types
# - Right: Harmonized JOINT embedding UMAP (CosMx cells only), same colors
# - Excludes these cell types entirely:
#     SST, VLMC, PVALB, ChC, uncertain
# Fixes your legend error by ensuring labels passed to fig.legend()
# are a plain Python list (NOT a numpy array / pandas Index).

# Objects
adata = bgs5
harm = harmonized

# Settings (edit if needed)
color_key = 'adjusted_L3_transferred'
batch_key = 'batch'
target_batch = '0'          # CosMx batch label inside harmonized.obs[batch_key]
id_col = 'cell_id'          # fallback matching key if obs_names don't match
exclude_types = {'SST', 'VLMC', 'PVALB', 'ChC', 'uncertain'}

# Plot styling
spatial_s = 1.0
umap_s = 0.6
alpha = 0.85

# Sanity checks
if 'X_umap' not in harm.obsm:
    raise ValueError("harmonized.obsm['X_umap'] not found. Run sc.tl.umap(harmonized) first.")
if color_key not in adata.obs.columns:
    raise KeyError(f"{color_key} not found in bgs5.obs")
if batch_key not in harm.obs.columns:
    raise KeyError(f"{batch_key} not found in harmonized.obs")

# Restrict harmonized to CosMx batch (UMAP coords for CosMx only)
harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

# Align harmonized CosMx UMAP rows to bgs5 cells
if np.array_equal(harm_cosmx_names.values, adata.obs_names.values):
    umap_coords = harm_umap_cosmx
else:
    # Fallback: match by id_col
    if id_col not in adata.obs.columns or id_col not in harm.obs.columns:
        raise ValueError(
            "CosMx cells in harmonized do not match bgs5.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match OR set id_col to a shared unique ID (e.g. 'cell_ID')."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    bgs5_obs = adata.obs[[id_col]].copy()
    bgs5_obs['__bgs5_i'] = np.arange(bgs5_obs.shape[0])

    merged = bgs5_obs.merge(harm_cosmx_obs, on=id_col, how='inner')

    if merged.shape[0] == 0:
        raise ValueError(
            f"No cells matched between bgs5 and harmonized CosMx cells using id_col='{id_col}'. "
            "Try switching id_col to 'cell_ID' (or whichever is truly unique and shared)."
        )

    if merged.shape[0] != adata.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata.n_obs:,} bgs5 cells to harmonized CosMx cells via '{id_col}'. "
            "Proceeding with matched subset only."
        )

    umap_coords = harm_umap_cosmx[merged['__umap_i'].values, :]
    adata = adata[merged['__bgs5_i'].values].copy()

# Filter out excluded cell types (apply to BOTH plots)
labels_all = adata.obs[color_key].astype(str)
mask_keep = ~labels_all.isin(exclude_types)

adata = adata[mask_keep].copy()
umap_coords = umap_coords[mask_keep.values, :]

labels = adata.obs[color_key].astype(str)

# Colors (prefer pre-defined)
unique_labels = np.sort(labels.unique())
unique_labels_list = list(map(str, unique_labels))  # <-- IMPORTANT: make plain python list

if (
    color_key in adata.obs and
    adata.obs[color_key].dtype.name == 'category' and
    f'{color_key}_colors' in adata.uns
):
    categories = adata.obs[color_key].cat.categories.astype(str)
    color_list = adata.uns[f'{color_key}_colors']
    label_colors_all = dict(zip(categories, color_list))

    # Restrict mapping to labels actually present
    label_colors = {k: label_colors_all.get(k, 'lightgrey') for k in unique_labels_list}
    print(f"Using pre-defined colors from adata.uns['{color_key}_colors']")
else:
    palette = sns.color_palette('tab20', len(unique_labels_list))
    label_colors = dict(zip(unique_labels_list, palette))

# Spatial coords
if 'spatial' in adata.obsm:
    xy = adata.obsm['spatial']
    x_sp, y_sp = xy[:, 0], xy[:, 1]
else:
    x_sp = adata.obs['CenterX_global_px'].to_numpy()
    y_sp = adata.obs['CenterY_global_px'].to_numpy()

# Plot: single figure, 2 panels
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Spatial (left)
sns.scatterplot(
    x=x_sp, y=y_sp,
    hue=labels.astype(str), palette=label_colors,
    s=spatial_s, edgecolor=None, ax=axes[0], legend=False, alpha=alpha
)
axes[0].set_title("CosMx Spatial (transferred cell types)")
axes[0].set_xlabel("X (px)")
axes[0].set_ylabel("Y (px)")
axes[0].invert_yaxis()
axes[0].set_aspect("equal")

# Harmonized UMAP (right) (CosMx only)
sns.scatterplot(
    x=umap_coords[:, 0], y=umap_coords[:, 1],
    hue=labels.astype(str), palette=label_colors,
    s=umap_s, edgecolor=None, ax=axes[1], legend=False, alpha=alpha
)
axes[1].set_title("Harmonized Joint UMAP (CosMx cells only)")
axes[1].set_xlabel("UMAP 1")
axes[1].set_ylabel("UMAP 2")

# Shared legend outside (FIXED)
# - pass labels=unique_labels_list (python list)
# - avoid numpy array truth-value ambiguity inside matplotlib
handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=label_colors[l], markersize=7, linestyle='None')
    for l in unique_labels_list
]

fig.legend(
    handles=handles,
    labels=unique_labels_list,
    title=color_key,
    bbox_to_anchor=(1.02, 0.5),
    loc='center left'
)

plt.tight_layout(rect=[0, 0, 0.86, 1])
plt.show()

In [ ]:
# saving harmonized with small file size

harm_slim = harmonized.copy()

# Drop expression data
harm_slim.X = None
harm_slim.layers.clear()
harm_slim.raw = None

# Drop unused embeddings
keep_obsm = {'X_umap'}  # keep ONLY the joint UMAP
harm_slim.obsm = {k: v for k, v in harm_slim.obsm.items() if k in keep_obsm}

# Drop large graph objects
harm_slim.obsp.clear()

# Drop variable-level info (not needed for plotting)
harm_slim.var = harm_slim.var.iloc[0:0]
harm_slim.varm.clear()

# Optional: clean uns (safe for plotting)
harm_slim.uns = {}

# import os

# outdir = "/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/"
# os.makedirs(outdir, exist_ok=True)

# # Save AnnData objects
# adata_filtered_600_leiden_filtered.write(os.path.join(outdir, "20260114_harmonized_joint_umap_slim.h5ad", compression="gzip"))

In [ ]:
import anndata as ad

# 1) Subset to 0 vars (this changes shape to (n_obs, 0))
harm_slim = harmonized[:, []].copy()

# 2) Drop big matrices / layers / raw
harm_slim.X = None
harm_slim.layers.clear()
harm_slim.raw = None
harm_slim.obsp.clear()
harm_slim.varm.clear()
harm_slim.uns = {}

# 3) Keep only UMAP embedding (and optionally other embeddings you want)
harm_slim.obsm = {'X_umap': harmonized.obsm['X_umap'].copy()}

# 4) Write
harm_slim.write("20260114_bgs5_3T_harmonized_joint_umap_slim.h5ad", compression="gzip")

In [ ]:
bgs5 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260108_bgs5_filtered_snm3_3T_L3.h5ad")

#3T-->bgs5
bgs5

In [ ]:
#earlier today saved the harmonized object umap only

harmonized = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260114_bgs5_3T_harmonized_joint_umap_slim.h5ad")
harmonized

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Rename batch categories
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Per-cell sizes
sizes = np.where(
    harmonized.obs['batch_named'] == 'CosMx',
    0.35,   # smaller CosMx
    1.4    # larger snm3C
)

# Plot WITHOUT legend
sc.pl.umap(
    harmonized,
    color='batch_named',
    size=sizes,
    palette={
        'CosMx': '#5DA5DA',
        'snm3C': '#D62728'
    },
    title="3T m3C  GW35 CosMx joint embedding",
    legend_loc=None,   # <-- suppress default dot legend
    show=False
)

ax = plt.gca()

# Build square legend handles
legend_handles = [
    mpatches.Patch(color='#5DA5DA', label='CosMx'),
    mpatches.Patch(color='#D62728', label='snm3C'),
]

ax.legend(
    handles=legend_handles,
    title="Batch",
    loc='best',
    frameon=False
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import scanpy as sc

# GLOBAL FONT SETTINGS (apply everywhere)
mpl.rcParams.update({
    'font.size': 18,          # base font
    'axes.titlesize': 24,     # plot title
    'axes.labelsize': 22,     # axis labels
    'xtick.labelsize': 18,    # tick labels
    'ytick.labelsize': 18,
    'legend.fontsize': 18,    # legend labels
    'legend.title_fontsize': 18
})

# Rename batch categories
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Per-cell sizes (CosMx smaller, snm3C larger)
sizes = np.where(
    harmonized.obs['batch_named'] == 'CosMx',
    0.35,   # CosMx (dense  smaller)
    1.5     # snm3C (sparser  larger)
)

# Plot UMAP WITHOUT default legend
sc.pl.umap(
    harmonized,
    color='batch_named',
    size=sizes,
    palette={
        'CosMx': '#5DA5DA',
        'snm3C': '#D62728'
    },
    title="3T m3C  GW35 CosMx joint embedding",
    legend_loc=None,   # suppress Scanpy dot legend
    show=False
)

ax = plt.gca()

# Custom square-color legend
legend_handles = [
    mpatches.Patch(color='#5DA5DA', label='CosMx'),
    mpatches.Patch(color='#D62728', label='snm3C-seq'),
]

ax.legend(
    handles=legend_handles,
    #title="Batch",
    loc='lower left',
    frameon=False,
    fontsize=13,
    title_fontsize=18
)

# Final polish
ax.set_title("3T snm3C and GW35 CosMx joint embedding", fontsize=16)

# remove frame / box around UMAP
ax.set_axis_off()          # removes spines + ticks + axis labels
# (optional) also ensure spines are off
for spine in ax.spines.values():
    spine.set_visible(False)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import scanpy as sc

# GLOBAL FONT + SVG SETTINGS (Illustrator-friendly)
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 24,
    'axes.labelsize': 22,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'legend.title_fontsize': 18,
    'svg.fonttype': 'none',   # keep text editable in SVG
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

# Rename batch categories
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Per-cell sizes (CosMx smaller, snm3C larger)
sizes = np.where(
    harmonized.obs['batch_named'] == 'CosMx',
    0.35,   # CosMx
    1.4     # snm3C
)

# Plot UMAP WITHOUT default Scanpy legend
sc.pl.umap(
    harmonized,
    color='batch_named',
    size=sizes,
    palette={
        'CosMx': '#5DA5DA',
        'snm3C': '#D62728'
    },
    title="3T m3C  GW35 CosMx joint embedding",
    legend_loc=None,
    show=False
)

ax = plt.gca()

# Rasterize the UMAP points (keeps SVG small/fast)
# Scanpy's scatter artists are stored in ax.collections
for coll in ax.collections:
    coll.set_rasterized(True)

# Custom square-color legend (top-right)
legend_handles = [
    mpatches.Patch(color='#5DA5DA', label='CosMx'),
    mpatches.Patch(color='#D62728', label='snm3C'),
]

ax.legend(
    handles=legend_handles,
    loc='upper right',
    frameon=False,
    fontsize=15
)

# Final polish
ax.set_title("3T snm3C and GW35 CosMx joint embedding", fontsize=17)
ax.tick_params(axis='both', which='major', labelsize=4)

# SAVE AS SVG (Illustrator-ready, rasterized points)
plt.savefig(
    "3T_m3C_GW35_CosMx_joint_embedding_batch.svg",
    bbox_inches="tight",
    dpi=600   # controls raster resolution of the points inside the SVG
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import scanpy as sc

# GLOBAL FONT + PDF SETTINGS (Arial + editable text in Illustrator)
mpl.rcParams.update({
    # Font
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],

    # Sizes
    'font.size': 18,
    'axes.titlesize': 24,
    'axes.labelsize': 22,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'legend.title_fontsize': 18,

    # Keep text as text (editable) in PDF
    'pdf.fonttype': 42,
    'ps.fonttype': 42,

    # If you ever save SVG too
    'svg.fonttype': 'none',
})

# Rename batch categories
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Per-cell sizes (CosMx smaller, snm3C larger)
sizes = np.where(
    harmonized.obs['batch_named'] == 'CosMx',
    0.35,   # CosMx (dense  smaller)
    1.5     # snm3C (sparser  larger)
)

# Plot UMAP WITHOUT default legend
sc.pl.umap(
    harmonized,
    color='batch_named',
    size=sizes,
    palette={
        'CosMx': '#5DA5DA',
        'snm3C': '#D62728'
    },
    title="3T m3C  GW35 CosMx joint embedding",
    legend_loc=None,
    show=False
)

ax = plt.gca()

# Rasterize ONLY the point cloud (small PDF), keep text/legend vector
for coll in ax.collections:
    coll.set_rasterized(True)

# Custom square-color legend (stays vector)
legend_handles = [
    mpatches.Patch(color='#5DA5DA', label='CosMx'),
    mpatches.Patch(color='#D62728', label='snm3C-seq'),
]

ax.legend(
    handles=legend_handles,
    loc='lower left',
    frameon=False,
    fontsize=13,
    title_fontsize=18
)

# Final polish
ax.set_title("3T snm3C and GW35 CosMx joint embedding", fontsize=16)

# remove frame / box around UMAP
ax.set_axis_off()
for spine in ax.spines.values():
    spine.set_visible(False)

# SAVE AS PDF (small file; dots rasterized; text stays editable)
out_pdf = "3T_GW35_umap_joint_batch_20260115.pdf"
plt.savefig(
    out_pdf,
    format="pdf",
    bbox_inches="tight",
    dpi=300,         # controls dot raster resolution; try 200 if you want even smaller
    transparent=True
)

plt.show()
print(f"Saved: {out_pdf}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches

# CosMx-only JOINT UMAP (coords from harmonized)
# - Colored by bgs5 transferred cell types
# - ONLY the specified cell types, in the specified order
# - Custom explicit colors
# - Illustrator-friendly SVG (editable text)
# - Rasterized points (small/fast SVG)
# - Legend OUTSIDE, very close, with RECTANGULAR swatches
# - Hide axis tick numbers
# - Bigger title

# Objects
adata = bgs5
harm = harmonized

# Required keys / settings
color_key = 'adjusted_L3_transferred'  # labels live in bgs5
batch_key = 'batch'                    # harmonized batch column
target_batch = '0'                     # CosMx inside harmonized
id_col = 'cell_id'                     # fallback matching key if obs_names differ

# Only keep these cell types + colors
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Aesthetics
umap_s = 0.1
alpha = 1
title_txt = "3T snm3C and GW35 CosMx joint embedding"

# GLOBAL FONT + SVG SETTINGS (Illustrator-friendly)
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 30,     # bigger title baseline
    'axes.labelsize': 14,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 12,
    'legend.title_fontsize': 14,
    'svg.fonttype': 'none',   # keep text editable in SVG
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

# Sanity checks
if 'X_umap' not in harm.obsm:
    raise ValueError("harmonized.obsm['X_umap'] not found.")
if color_key not in adata.obs.columns:
    raise KeyError(f"{color_key} not found in bgs5.obs")
if batch_key not in harm.obs.columns:
    raise KeyError(f"{batch_key} not found in harmonized.obs")

# Restrict harmonized to CosMx batch (UMAP coords for CosMx only)
harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

# Align harmonized CosMx UMAP rows to bgs5 cells
if np.array_equal(harm_cosmx_names.values, adata.obs_names.values):
    umap_coords = harm_umap_cosmx
else:
    if id_col not in adata.obs.columns or id_col not in harm.obs.columns:
        raise ValueError(
            "CosMx cells in harmonized do not match bgs5.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match OR set id_col to a shared unique ID (e.g. 'cell_ID')."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    bgs5_obs = adata.obs[[id_col]].copy()
    bgs5_obs['__bgs5_i'] = np.arange(bgs5_obs.shape[0])

    merged = bgs5_obs.merge(harm_cosmx_obs, on=id_col, how='inner')

    if merged.shape[0] == 0:
        raise ValueError(
            f"No cells matched between bgs5 and harmonized CosMx cells using id_col='{id_col}'. "
            "Try switching id_col to 'cell_ID' (or whichever is truly unique and shared)."
        )

    if merged.shape[0] != adata.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata.n_obs:,} bgs5 cells to harmonized CosMx cells via '{id_col}'. "
            "Proceeding with matched subset only."
        )

    umap_coords = harm_umap_cosmx[merged['__umap_i'].values, :]
    adata = adata[merged['__bgs5_i'].values].copy()

# Keep ONLY specified cell types (and order legend accordingly)
labels_all = adata.obs[color_key].astype(str)
keep_set = set(cell_type_order)
mask_keep = labels_all.isin(keep_set)

adata = adata[mask_keep].copy()
umap_coords = umap_coords[mask_keep.values, :]
labels = adata.obs[color_key].astype(str)

# Ensure legend order is exactly cell_type_order, but only those present
present = [ct for ct in cell_type_order if ct in set(labels.unique())]

# Colors for present types (fallback to lightgrey if missing)
label_colors = {ct: cell_type_colors.get(ct, 'lightgrey') for ct in present}

# Convert labels -> per-point colors in correct mapping
point_colors = [label_colors.get(l, 'lightgrey') for l in labels.values]

# RANDOMIZE POINT DRAW ORDER
rng = np.random.default_rng(0)  # set seed for reproducibility; remove seed for true randomness
perm = rng.permutation(umap_coords.shape[0])

umap_coords = umap_coords[perm]
point_colors = np.array(point_colors)[perm]

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

scat = ax.scatter(
    umap_coords[:, 0], umap_coords[:, 1],
    c=point_colors,
    s=umap_s,
    alpha=alpha,
    linewidths=0
)

# Rasterize points (keeps SVG small/fast)
scat.set_rasterized(True)

# Title bigger
ax.set_title(title_txt, fontsize=17)

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

# Hide tick numbers (keep clean axes)
ax.set_xticks([])
ax.set_yticks([])
ax.tick_params(bottom=False, left=False)

# Optional: remove top/right spines for cleaner look
# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)

# Legend OUTSIDE but very close + rectangular swatches
legend_handles = [
    mpatches.Patch(facecolor=label_colors[ct], edgecolor='none', label=ct)
    for ct in present
]

ax.legend(
    handles=legend_handles,
    #title=color_key,
    loc='center left',
    bbox_to_anchor=(1.005, 0.5),  # very close to axes
    frameon=False,
    borderaxespad=0.0,
    handlelength=1.2,
    handleheight=0.9,
)

# Leave a bit of room on the right for the close legend
plt.tight_layout(rect=[0, 0, 0.86, 1])

# SAVE AS SVG (Illustrator-ready, rasterized points)
plt.savefig(
    "GW35_CosMx_joint_embedding_celltypes_filtered.svg",
    bbox_inches="tight",
    dpi=600
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches

# CosMx-only JOINT UMAP (coords from harmonized)
# - Colored by bgs5 transferred cell types
# - ONLY the specified cell types, in the specified order
# - Custom explicit colors
# - Arial + editable text in Illustrator (PDF)
# - Rasterized points (small PDF)
# - Legend OUTSIDE, very close, with rectangular swatches
# - Hide axis tick numbers
# - Remove ALL spines/frame

# Objects
adata = bgs5
harm = harmonized

# Required keys / settings
color_key = 'adjusted_L3_transferred'  # labels live in bgs5
batch_key = 'batch'                    # harmonized batch column
target_batch = '0'                     # CosMx inside harmonized
id_col = 'cell_id'                     # fallback matching key if obs_names differ

# Only keep these cell types + colors
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Aesthetics
umap_s = 0.1
alpha = 1
title_txt = "3T snm3C and GW35 CosMx joint embedding"

# GLOBAL FONT + PDF SETTINGS (Arial + editable text in Illustrator)
mpl.rcParams.update({
    # Arial everywhere
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],

    'font.size': 18,
    'axes.titlesize': 30,
    'axes.labelsize': 14,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 12,
    'legend.title_fontsize': 14,

    # Keep text editable in PDF
    'pdf.fonttype': 42,
    'ps.fonttype': 42,

    # If you ever save SVG too
    'svg.fonttype': 'none',
})

# Sanity checks
if 'X_umap' not in harm.obsm:
    raise ValueError("harmonized.obsm['X_umap'] not found.")
if color_key not in adata.obs.columns:
    raise KeyError(f"{color_key} not found in bgs5.obs")
if batch_key not in harm.obs.columns:
    raise KeyError(f"{batch_key} not found in harmonized.obs")

# Restrict harmonized to CosMx batch (UMAP coords for CosMx only)
harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

# Align harmonized CosMx UMAP rows to bgs5 cells
if np.array_equal(harm_cosmx_names.values, adata.obs_names.values):
    umap_coords = harm_umap_cosmx
else:
    if id_col not in adata.obs.columns or id_col not in harm.obs.columns:
        raise ValueError(
            "CosMx cells in harmonized do not match bgs5.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match OR set id_col to a shared unique ID (e.g. 'cell_ID')."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    bgs5_obs = adata.obs[[id_col]].copy()
    bgs5_obs['__bgs5_i'] = np.arange(bgs5_obs.shape[0])

    merged = bgs5_obs.merge(harm_cosmx_obs, on=id_col, how='inner')

    if merged.shape[0] == 0:
        raise ValueError(
            f"No cells matched between bgs5 and harmonized CosMx cells using id_col='{id_col}'. "
            "Try switching id_col to 'cell_ID' (or whichever is truly unique and shared)."
        )

    if merged.shape[0] != adata.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata.n_obs:,} bgs5 cells to harmonized CosMx cells via '{id_col}'. "
            "Proceeding with matched subset only."
        )

    umap_coords = harm_umap_cosmx[merged['__umap_i'].values, :]
    adata = adata[merged['__bgs5_i'].values].copy()

# Keep ONLY specified cell types (and order legend accordingly)
labels_all = adata.obs[color_key].astype(str)
keep_set = set(cell_type_order)
mask_keep = labels_all.isin(keep_set)

adata = adata[mask_keep].copy()
umap_coords = umap_coords[mask_keep.values, :]
labels = adata.obs[color_key].astype(str)

# Ensure legend order is exactly cell_type_order, but only those present
present = [ct for ct in cell_type_order if ct in set(labels.unique())]

# Colors for present types (fallback to lightgrey if missing)
label_colors = {ct: cell_type_colors.get(ct, 'lightgrey') for ct in present}

# Convert labels -> per-point colors
point_colors = [label_colors.get(l, 'lightgrey') for l in labels.values]

# RANDOMIZE POINT DRAW ORDER
rng = np.random.default_rng(0)
perm = rng.permutation(umap_coords.shape[0])

umap_coords = umap_coords[perm]
point_colors = np.array(point_colors)[perm]

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

scat = ax.scatter(
    umap_coords[:, 0], umap_coords[:, 1],
    c=point_colors,
    s=umap_s,
    alpha=alpha,
    linewidths=0
)

# Rasterize points (keeps PDF small)
scat.set_rasterized(True)

# Title
ax.set_title(title_txt, fontsize=17)

#ax.set_xlabel("")
#ax.set_ylabel("")

# Hide tick numbers (keep clean axes)
ax.set_xticks([])
ax.set_yticks([])
ax.tick_params(bottom=False, left=False)

# REMOVE ALL SPINES / FRAME
for spine in ax.spines.values():
    spine.set_visible(False)

# Legend OUTSIDE but very close + rectangular swatches
legend_handles = [
    mpatches.Patch(facecolor=label_colors[ct], edgecolor='none', label=ct)
    for ct in present
]

ax.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(1.005, 0.5),
    frameon=False,
    borderaxespad=0.0,
    handlelength=1.2,
    handleheight=0.9,
)

# Leave a bit of room on the right for the close legend
plt.tight_layout(rect=[0, 0, 0.86, 1])

# SAVE AS PDF (Illustrator-ready: editable text + small file)
out_pdf = "GW35_CosMx_joint_embedding_celltypes_filtered.pdf"
plt.savefig(
    out_pdf,
    format="pdf",
    bbox_inches="tight",
    dpi=300,         # raster layer resolution; drop to 200 for smaller
    transparent=True
)

plt.show()
print(f"Saved: {out_pdf}")